# Activation PCA and saved clusters from one path

Set only `ANALYSIS_PATH`: a DALG activation-shard directory, an experiment folder,
or an individual pipeline run (or its `run_spec.json` / model checkpoint).
Saved run metadata supplies the activation path and layers, including when models
live separately from their data. A directory containing multiple datasets is
ambiguous: choose one dataset's experiment instead.

Prompt labels are discovered from metadata; no dataset names or label names are
required. Other DALG shard datasets use any row-level `labels` metadata, or show
neutral points when labels are absent. Windowed data includes every retained token,
respecting `drop_prefix`. Numeric and categorical colors are supported.
PCA centers features without scaling or whitening. Each model panel has its own
run selector, so seeds can be inspected without changing paths or refitting PCA.


In [26]:
from pathlib import Path
from collections import defaultdict
from contextlib import nullcontext
import json
import os
import sys
from time import perf_counter

import numpy as np
import pandas as pd
import torch
import k3d
from sklearn.decomposition import PCA
from sklearn.metrics import homogeneity_score
from joblib import Parallel, delayed, parallel_config
from IPython.display import display

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))
from dalg.data.prompt_activations import load_prompt_activation_index
from dalg.data.shard_activations import load_meta_index
from dalg.data.subset_spec import split_shard_dir_spec, resolve_spec_positions
from dalg.models.kmeans import load_kmeans
from dalg.models.adaptive_q.mfa_hddc import load_mfa_hddc

# The only path to edit. Relative paths are relative to the repository.
ANALYSIS_PATH = "dalg-cache/sae-manifold/sae_manifold_years_neutral_llama31_8b_last_SEED_EXPLORATION"
HOMOGENEITY_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
HOMOGENEITY_BATCH_SIZE = 512


def resolve_analysis_path(path):
    path = Path(path).expanduser()
    path = (REPO_ROOT / path).resolve() if not path.is_absolute() else path.resolve()
    if path.is_file():
        path = path.parent
    if not path.is_dir():
        raise FileNotFoundError(path)
    runs = []
    for spec_path in sorted(path.rglob("run_spec.json")):
        spec = json.loads(spec_path.read_text())
        if spec["training"]["model_kind"] in ("kmeans", "hddc"):
            runs.append((spec_path, spec))
    datasets = set()
    for _, spec in runs:
        root, subset = split_shard_dir_spec(spec["dataset"]["shard_dir"])
        root = (REPO_ROOT / root).resolve()
        prefix = spec["training"]["arguments"].get("drop_prefix")
        if prefix is None:
            prefix = json.loads((root / "config.json").read_text()).get("drop_prefix", 0)
        datasets.add((root, subset, int(prefix)))
    is_activation_root = (path / "config.json").is_file() and any(path.glob("layer[0-9]*"))
    if len(datasets) > 1:
        raise ValueError("Choose a path with one activation dataset, subset, and prefix policy.")
    if datasets:
        activation_dir, subset_spec, drop_prefix = next(iter(datasets))
        if is_activation_root and activation_dir != path:
            raise ValueError("Saved runs refer to a different activation dataset than this directory.")
        layers = sorted({int(spec["dataset"]["layer"]) for _, spec in runs})
    elif is_activation_root:
        activation_dir, subset_spec, drop_prefix = path, None, None
        layers = sorted(int(folder.name[5:]) for folder in path.glob("layer[0-9]*") if folder.is_dir())
    else:
        raise ValueError(f"No supported pipeline run_spec.json or activation shards under {path}")
    config = json.loads((activation_dir / "config.json").read_text())
    drop_prefix = int(config.get("drop_prefix", 0) if drop_prefix is None else drop_prefix)
    if not 0 <= drop_prefix < int(config["window"]):
        raise ValueError("drop_prefix must leave at least one token per row")
    for layer in layers:
        if not (activation_dir / f"layer{layer:02d}").is_dir():
            raise FileNotFoundError(activation_dir / f"layer{layer:02d}")
    return path, runs, activation_dir, subset_spec, drop_prefix, layers, config


MODEL_ROOT, RUN_SPECS, ACTIVATION_DIR, SUBSET_SPEC, DROP_PREFIX, LAYERS, config = resolve_analysis_path(ANALYSIS_PATH)


def activation_metadata(layer):
    coordinates = load_meta_index(ACTIVATION_DIR, layer)
    positions = resolve_spec_positions(coordinates, SUBSET_SPEC, window=config["window"], drop_prefix=DROP_PREFIX)
    if config.get("format") == "dalg_prompt_activations_v1":
        index = load_prompt_activation_index(ACTIVATION_DIR, layer)
    else:
        shard_metadata = {}
        index = []
        for coordinate in coordinates:
            shard = coordinate["shard"]
            if shard not in shard_metadata:
                shard_metadata[shard] = json.loads((ACTIVATION_DIR / "meta" / f"shard_{shard:05d}.json").read_text())["rows"]
            attributes = shard_metadata[shard][coordinate["row_in_shard"]]
            index.append({**attributes, **coordinate})
    return pd.DataFrame([
        {**index[position], "token_slot": token, "labels": index[position].get("labels", {})}
        for position in positions for token in range(DROP_PREFIX, int(config["window"]))
    ])


metadata = activation_metadata(LAYERS[0])
SUBSETS = sorted(metadata["subset"].unique())
summary = metadata.groupby("subset").size().rename("activations").to_frame()
summary["label_fields"] = [
    sorted({key for labels in metadata.loc[metadata.subset.eq(name), "labels"] for key in labels})
    for name in summary.index
]
print(f"Data: {ACTIVATION_DIR}\nModels: {MODEL_ROOT}")
print(f"{len(metadata):,} activations; layers {LAYERS}; {config['d_model']} features")
display(summary)


Data: /orfeo/cephfs/scratch/dssc/zenocosini/dalg-cache/sae-manifold/sae_manifold_years_neutral_llama31_8b_last
Models: /orfeo/cephfs/scratch/dssc/zenocosini/dalg-cache/sae-manifold/sae_manifold_years_neutral_llama31_8b_last_SEED_EXPLORATION
16,016 activations; layers [4]; 4096 features


,activations,label_fields
subset,,
years,16016,"[decade, decade_digit, template_id, year]"


## Available models: choose a checkpoint

`available_models` lists every existing model checkpoint discovered under
`ANALYSIS_PATH`, independently of the plot selections and optional plot filters.
Choose a row's `model_index` below and run the loading cell to get the full CPU
`model`. No PCA or plot cells are needed. Change `MODEL_INDEX` and rerun to load
another checkpoint. Component IDs are zero-based and belong to that checkpoint.


In [27]:
available_models = []
for spec_path, spec in RUN_SPECS:
    kind = spec["training"]["model_kind"]
    args = spec["training"]["arguments"]
    checkpoint = spec_path.parent / ("mfa_model.pt" if kind == "hddc" else "kmeans_model.pt")
    if checkpoint.is_file():
        available_models.append({
            "kind": kind,
            "layer": int(spec["dataset"]["layer"]),
            "K": args["K"],
            "seed": args["seed"],
            "threshold": args["surgery_threshold"],
            "run": spec_path.parent.name,
            "checkpoint": checkpoint,
        })

available_model_table = pd.DataFrame(available_models)
available_model_table.index.name = "model_index"
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(available_model_table)
print(f"{len(available_models)} checkpoints available. Choose MODEL_INDEX in the next cell.")


,kind,layer,K,seed,threshold,run,checkpoint
model_index,,,,,,,
0,hddc,4,50,42,0.01,hddc__years_neutral__l04__k50__q16__s42__f7c18d6a,/orfeo/cephfs/scratch/dssc/zenocosini/dalg-cache/sae-manifold/sae_manifold_years_neutral_llama31_8b_last_SEED_EXPLORATION/sae_manifold_years_neutral_hddc_em_seed_exploration/hddc__years_neutral__l04__k50__q16__s42__f7c18d6a/mfa_model.pt
1,hddc,4,50,43,0.01,hddc__years_neutral__l04__k50__q16__s43__75842954,/orfeo/cephfs/scratch/dssc/zenocosini/dalg-cache/sae-manifold/sae_manifold_years_neutral_llama31_8b_last_SEED_EXPLORATION/sae_manifold_years_neutral_hddc_em_seed_exploration/hddc__years_neutral__l04__k50__q16__s43__75842954/mfa_model.pt
2,hddc,4,50,44,0.01,hddc__years_neutral__l04__k50__q16__s44__3563c899,/orfeo/cephfs/scratch/dssc/zenocosini/dalg-cache/sae-manifold/sae_manifold_years_neutral_llama31_8b_last_SEED_EXPLORATION/sae_manifold_years_neutral_hddc_em_seed_exploration/hddc__years_neutral__l04__k50__q16__s44__3563c899/mfa_model.pt
3,hddc,4,50,45,0.01,hddc__years_neutral__l04__k50__q16__s45__0b12d30a,/orfeo/cephfs/scratch/dssc/zenocosini/dalg-cache/sae-manifold/sae_manifold_years_neutral_llama31_8b_last_SEED_EXPLORATION/sae_manifold_years_neutral_hddc_em_seed_exploration/hddc__years_neutral__l04__k50__q16__s45__0b12d30a/mfa_model.pt
4,hddc,4,50,46,0.01,hddc__years_neutral__l04__k50__q16__s46__93442129,/orfeo/cephfs/scratch/dssc/zenocosini/dalg-cache/sae-manifold/sae_manifold_years_neutral_llama31_8b_last_SEED_EXPLORATION/sae_manifold_years_neutral_hddc_em_seed_exploration/hddc__years_neutral__l04__k50__q16__s46__93442129/mfa_model.pt


5 checkpoints available. Choose MODEL_INDEX in the next cell.


In [217]:
MODEL_INDEX = 4  # choose a model_index from the table above
COMPONENT_ID = 1  # component within this checkpoint

if not available_models:
    raise ValueError("No saved model checkpoints were found under ANALYSIS_PATH.")
selected_checkpoint = available_models[MODEL_INDEX]
KIND = selected_checkpoint["kind"]
loader = load_mfa_hddc if KIND == "hddc" else load_kmeans
model = loader(selected_checkpoint["checkpoint"], map_location="cpu")
if not 0 <= COMPONENT_ID < model.K:
    raise ValueError(f"Choose a component ID from 0 to {model.K - 1}.")

print(f"Layer: {selected_checkpoint['layer']} | Checkpoint: {selected_checkpoint['checkpoint']}")
print(f"Model: {type(model).__name__} | K={model.K} | D={model.D}")
print(f"Component {COMPONENT_ID} | saved rank={int(model.component_ranks[COMPONENT_ID])}")
display(model)
display(model.mu[COMPONENT_ID])  # centroid in the original activation space

# Further inspection in another cell:
# HDDC: model._W()[COMPONENT_ID], model._psi()[COMPONENT_ID]
#       model.pi_logits.softmax(dim=0)[COMPONENT_ID]
# KMeans: model.W[COMPONENT_ID], model.eigenvalues[COMPONENT_ID]
#         model.cluster_counts[COMPONENT_ID]


Layer: 4 | Checkpoint: /orfeo/cephfs/scratch/dssc/zenocosini/dalg-cache/sae-manifold/sae_manifold_years_neutral_llama31_8b_last_SEED_EXPLORATION/sae_manifold_years_neutral_hddc_em_seed_exploration/hddc__years_neutral__l04__k50__q16__s46__93442129/mfa_model.pt
Model: MFA_HDDC | K=50 | D=4096
Component 1 | saved rank=8


MFA_HDDC()

tensor([ 0.0216, -0.0461,  0.0377,  ..., -0.0071, -0.0151,  0.0663],
       grad_fn=<SelectBackward0>)

In [ ]:
display(model.mu[1])
display(model.component_ranks)



tensor([ 0.0361, -0.0408,  0.0182,  ...,  0.0043,  0.0179,  0.0613],
       grad_fn=<SelectBackward0>)

tensor([16, 11, 14, 16, 14, 16, 15, 16, 16, 12, 15, 12, 11, 16, 16, 16, 16, 16,
        16, 15, 15, 12, 16, 13,  5, 15, 15, 16, 16, 16, 16,  7, 14, 10, 12, 16,
        14, 10, 15, 16, 16, 15, 16, 15, 16, 10, 16, 16, 14, 15])

In [ ]:
compute_flattness(model, 1)

tensor(0.0003, grad_fn=<DivBackward0>)

In [207]:
def compute_flattness(model, component_id):
    cov = model._W()[component_id] @ model._W()[component_id].T + torch.diag(model._psi()[component_id])
    eigenvalues, _ = torch.linalg.eigh(cov)
    flattness = eigenvalues[0] / eigenvalues[-1]
    return flattness

flattness_values = []
for n, i in enumerate(range(model.K)):
    i = compute_flattness(model, i)
    flattness_values.append(i)
    print(f"Component {n}: Flattness = {i:.6f}")


Component 0: Flattness = 0.000035
Component 1: Flattness = 0.000193


KeyboardInterrupt: 

In [178]:
flattness_values = torch.tensor(flattness_values)
flattness_mask = flattness_values < 1e-3

In [208]:
# Compute NLL
layer = selected_checkpoint["layer"]
X = load_subset_layer(activation_metadata(layer), layer)

k = 1
components_nll = []
for k in range(model.K):
    with torch.no_grad():
        x = torch.as_tensor(X[:512], device=model.mu.device, dtype=model.mu.dtype)
        component_nll = -model.log_prob_components(x)[:, k].mean()
        components_nll.append(component_nll.item())

    print(component_nll.item())

components_nll = torch.tensor(components_nll)


104197.046875
956.764892578125
137614.328125
63951.18359375
83953.9765625
12322.302734375
4881.71484375
70304.90625
46876.72265625
75340.9921875
14486.3466796875
27997.158203125
76161.3203125
36758.9296875
10610.07421875
34865.875
7259.47607421875
86335.2890625
21269.876953125
-483.448974609375
80203.7265625
92615.65625
85240.59375
80982.2578125
123188.3828125
138684.0
7724.1083984375
1137.5008544921875
7483.5146484375
2335.489990234375
21391.353515625
77072.453125
-3990.403564453125
48065.8671875
100488.15625
4266.51171875
15158.154296875
19237.767578125
128523.9140625
61500.08203125
15916.09375
4283.9931640625
6452.1943359375
11836.865234375
3741.45556640625
64830.453125
27301.3046875
120789.421875
1042.912841796875
14591.75390625


In [218]:
with torch.no_grad(), model.inference_cache(
    dtype=torch.float64, component_chunk_size=8
):
    logp = torch.cat([
        model.log_prob_components(
            torch.as_tensor(
                X[start:start + 512],
                device=model.mu.device,
                dtype=model.mu.dtype,
            )
        ).cpu()
        for start in range(0, len(X), 512)
    ])

log_pi = model.pi_logits.detach().cpu().double().log_softmax(0)
assigned = (logp + log_pi).argmax(dim=1)

components_nll = -logp.mean(dim=0)
assigned_nll = torch.stack([
    -logp[assigned == k, k].mean()  # NaN if no points are assigned
    for k in range(model.K)
])

In [219]:
torch.sort(assigned_nll, descending=True)

torch.return_types.sort(
values=tensor([-10440.6993, -10715.6909, -10770.3852, -10823.0000, -11004.6229,
        -11158.5332, -11187.5111, -11209.5764, -11289.5340, -11345.3389,
        -11582.6825, -11617.0543, -11768.2115, -11822.0723, -11853.2519,
        -11989.2476, -12061.0662, -12202.4017, -12317.2696, -12393.8054,
        -12424.1028, -12475.1860, -12710.6368, -12762.7301, -12839.6267,
        -12853.9893, -12875.7128, -12934.0571, -12999.3238, -13224.5164,
        -13844.6576, -13872.3098, -13937.4255, -14037.6624, -14104.3174,
        -14134.7592, -14157.8406, -14180.8936, -14181.6040, -14183.1097,
        -14319.0339, -14369.4799, -14393.1316, -14416.9155, -14482.6781,
        -14602.4455, -14724.7181, -14811.6679, -14848.7374, -14857.7210],
       dtype=torch.float64),
indices=tensor([24, 19, 22, 41, 40, 38, 14, 37,  8, 25, 45, 27,  3,  7, 12, 18, 31, 43,
         9, 30, 33, 15, 16, 39, 29, 21,  4, 42,  2, 47, 35, 49,  6, 44,  5, 23,
        46, 17, 34,  0, 28, 10, 32, 11, 

## Optional subset and coloring

`SUBSET = None` uses all rows selected by the saved run. Set a subset name from
the table to narrow the view. `COLOR_LABEL = None` chooses the first available
label field; set a field explicitly to change it. Missing labels have a neutral
color and are excluded from homogeneity. Without labels, all points are neutral
and homogeneity is unavailable. Categorical colors include a legend.

Rerun selection and subsequent cells after changing these settings.


In [29]:
SUBSET = None
COLOR_LABEL = None
SEED = 1
N_COMPONENTS = 15
N_WORKERS = min(8, len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity")
                else (os.cpu_count() or 1))

if SUBSET is not None and SUBSET not in SUBSETS:
    raise ValueError(f"Choose SUBSET from {SUBSETS}, or None for all")
selected = metadata.loc[metadata.subset.eq(SUBSET)].reset_index(drop=True) if SUBSET is not None else metadata.copy()
labels = pd.json_normalize(selected["labels"].tolist())
color_label = COLOR_LABEL if COLOR_LABEL is not None else next(iter(labels.columns), None)
if color_label is not None and color_label not in labels:
    raise ValueError(f"Choose COLOR_LABEL from {labels.columns.tolist()}")
selection_name = SUBSET or ACTIVATION_DIR.name
print(f"Selected {selection_name}: {len(selected):,} activations; color = {color_label or 'unlabeled'}")
preview_columns = [name for name in ("global_row", "token_slot", "subset", "prompt", "decoded_token") if name in selected]
display(pd.concat([selected[preview_columns], labels], axis=1).head())


def point_coloring():
    colors = np.full(len(selected), 0x999999, dtype=np.uint32)
    if color_label is None:
        return colors, "Unlabeled points"
    values = labels[color_label]
    present = values.notna()
    numeric = pd.to_numeric(values, errors="coerce")
    if numeric[present].notna().all() and np.isfinite(numeric[present].to_numpy(dtype=float)).all():
        valid = present.to_numpy()
        numbers = numeric[present].to_numpy(dtype=float)
        if len(numbers):
            lo, hi = float(numbers.min()), float(numbers.max())
            colors[valid] = k3d.helpers.map_colors(numbers, k3d.colormaps.matplotlib_color_maps.Viridis,
                                                  [lo, hi if hi > lo else lo + 1])
            legend = f"{color_label}: {lo:g} (purple) → {hi:g} (yellow)"
        else:
            legend = f"{color_label}: no labels"
    else:
        codes, categories = pd.factorize(values[present].map(str), sort=True)
        palette = k3d.helpers.map_colors(np.arange(len(categories), dtype=float),
                    k3d.colormaps.matplotlib_color_maps.Turbo, [0, max(1, len(categories) - 1)])
        colors[present.to_numpy()] = palette[codes]
        from html import escape
        legend = f"{escape(color_label)}: " + " · ".join(
            f'<span style="color:#{int(color):06x}">{escape(str(value))}</span>'
            for value, color in zip(categories, palette))
    return colors, legend + " · gray = missing label"


Selected sae_manifold_years_neutral_llama31_8b_last: 16,016 activations; color = year


,global_row,token_slot,subset,prompt,decoded_token,year,decade_digit,decade,template_id
0,0,0,years,The year is 1000,0,1000.0,0.0,0.0,0
1,1,0,years,"According to this record, the year is 1000",0,1000.0,0.0,0.0,1
2,2,0,years,"For this entry, the year is 1000",0,1000.0,0.0,0.0,2
3,3,0,years,"In this account, the year is 1000",0,1000.0,0.0,0.0,3
4,4,0,years,"As noted here, the year is 1000",0,1000.0,0.0,0.0,4


## Fit 15 PCs for every layer in parallel

`N_WORKERS` controls how many layers run concurrently (up to eight by default).
Each worker loads only the selected subset, uses one CPU thread, and fits all its
rows, including distinct records with duplicate prompt text. A seeded randomized
SVD estimates the top 15 PCs with 20 oversamples and 7 power iterations.

Completed layers report loading and PCA times immediately, in completion order.
The final tables and layer selector remain ordered by layer number. Each worker
holds one activation matrix, so memory use increases with `N_WORKERS`.
`results` retains scores, PCA models, and loading/fitting times. No extraction
artifacts are modified.

In [30]:
def load_subset_layer(rows, layer):
    """Read selected rows in their metadata order, grouping reads by saved shard."""
    groups = defaultdict(list)
    for output_row, row in enumerate(rows.itertuples(index=False)):
        groups[int(row.shard)].append((output_row, int(row.row_in_shard), int(row.token_slot)))
    values = np.empty((len(rows), config["d_model"]), dtype=np.float32)
    for shard_id, coordinates in groups.items():
        path = ACTIVATION_DIR / f"layer{layer:02d}" / f"shard_{shard_id:05d}.pt"
        shard = torch.load(path, map_location="cpu", weights_only=True, mmap=True)
        if shard.ndim != 3 or tuple(shard.shape[1:]) != (config["window"], config["d_model"]):
            raise ValueError(f"Unexpected activation shape in {path}: {tuple(shard.shape)}")
        output_rows, shard_rows, token_slots = map(list, zip(*coordinates))
        values[output_rows] = shard[shard_rows, token_slots].float().numpy()
    if not np.isfinite(values).all():
        raise ValueError(f"Nonfinite activations in layer {layer}")
    return values


def fit_layer(layer, rows, seed):
    """Load and project one layer; joblib limits each worker to one CPU thread."""
    started = perf_counter()
    layer_rows = activation_metadata(layer)
    if SUBSET is not None:
        layer_rows = layer_rows.loc[layer_rows.subset.eq(SUBSET)].reset_index(drop=True)
    keys = ["global_row", "token_slot"]
    if not layer_rows[keys].reset_index(drop=True).equals(rows[keys].reset_index(drop=True)):
        raise ValueError(f"Layer {layer} has different activation coordinates")
    x = load_subset_layer(layer_rows, layer)
    loaded = perf_counter()
    if not np.any(x != x[0]):
        raise ValueError(f"Layer {layer} has zero variance across the selected prompts")
    pca = PCA(n_components=N_COMPONENTS, svd_solver="randomized", random_state=seed,
              n_oversamples=20, iterated_power=7)
    scores = pca.fit_transform(x)
    fitted = perf_counter()
    result = {
        "scores": scores, "pca": pca,
        "load_seconds": loaded - started, "pca_seconds": fitted - loaded,
    }
    return layer, result


if min(len(selected) - 1, config["d_model"]) < N_COMPONENTS:
    raise ValueError(f"{N_COMPONENTS} centered PCs require at least {N_COMPONENTS + 1} prompts and {N_COMPONENTS} features")
if not isinstance(N_WORKERS, int) or N_WORKERS < 1:
    raise ValueError("N_WORKERS must be a positive integer")
results = {}
workers = min(N_WORKERS, len(LAYERS))
started = perf_counter()
print(f"Fitting {len(LAYERS)} layers with {workers} parallel workers...", flush=True)
# Separate processes avoid competing BLAS/OpenMP thread pools in the kernel.
with parallel_config(backend="loky", inner_max_num_threads=1):
    completed = Parallel(n_jobs=workers, return_as="generator_unordered")(
        delayed(fit_layer)(layer, selected, SEED) for layer in LAYERS
    )
    for layer, result in completed:
        results[layer] = result
        ratios = result["pca"].explained_variance_ratio_
        print(f"[{len(results):02d}/{len(LAYERS)}] Layer {layer:02d}: "
              f"top {N_COMPONENTS} explain {ratios.sum():.2%}; "
              f"load {result['load_seconds']:.2f}s, PCA {result['pca_seconds']:.2f}s",
              flush=True)
results = dict(sorted(results.items()))
variance = pd.DataFrame({
    layer: result["pca"].explained_variance_ratio_
    for layer, result in results.items()
}, index=[f"PC{i + 1}" for i in range(N_COMPONENTS)]).T
variance.index.name = "layer"
print(f"Finished in {perf_counter() - started:.1f}s", flush=True)


Fitting 1 layers with 1 parallel workers...
[01/1] Layer 04: top 15 explain 40.43%; load 1.10s, PCA 2.30s
Finished in 3.4s


## Saved cluster models

All model discovery uses `ANALYSIS_PATH`. Optional K, threshold, and seed filters
below narrow the plot choices. Each plot's **Run** selector handles multiple
matching configurations, including initialization seeds. Missing checkpoints stay
listed in the summary and are omitted from plot choices.


## Component summary

Scan the KMeans and HDDC runs under the single analysis path. Alive components
have finite HDDC mixture logits or positive KMeans training counts. Mean rank uses
live components, excluding invalid KMeans PCA ranks.

Homogeneity uses the selected label field and subset, with exact numeric values
or categorical values treated as classes. Unlabeled points are excluded. No label
means no homogeneity score. Models are scored in the original activation space,
not PCA coordinates. This cell performs inference for every available checkpoint.


In [31]:
def component_summary(model, kind):
    ranks = model.component_ranks.detach().cpu()
    if kind == "hddc":
        # Zero-weight EM components have -inf logits; avoid softmax underflow.
        alive = torch.isfinite(model.pi_logits.detach().cpu())
        rank_eligible = alive
    elif kind == "kmeans":
        alive = model.cluster_counts.detach().cpu() > 0
        rank_eligible = alive & model.pca_valid.detach().cpu()
    else:
        raise ValueError(f"Unsupported model kind: {kind}")
    rank_count = int(rank_eligible.sum())
    return {
        "components_alive": int(alive.sum()),
        "components_total": int(model.K),
        "mean_q_k": float(ranks[rank_eligible].double().mean()) if rank_count else np.nan,
        "rank_components": rank_count,
    }


def model_homogeneity(model, kind, layer):
    if color_label is None:
        return np.nan
    valid = labels[color_label].notna().to_numpy()
    if not valid.any():
        return np.nan
    rows = activation_metadata(layer)
    if SUBSET is not None:
        rows = rows.loc[rows.subset.eq(SUBSET)].reset_index(drop=True)
    keys = ["global_row", "token_slot"]
    if not rows[keys].reset_index(drop=True).equals(selected[keys].reset_index(drop=True)):
        raise ValueError(f"Layer {layer} has different activation coordinates")
    x = load_subset_layer(rows.loc[valid], layer)
    model.to(HOMOGENEITY_DEVICE).eval()
    cache = model.inference_cache() if kind == "hddc" else nullcontext()
    assignments = []
    with torch.inference_mode(), cache:
        for start in range(0, len(x), HOMOGENEITY_BATCH_SIZE):
            batch = torch.as_tensor(x[start:start + HOMOGENEITY_BATCH_SIZE], device=model.mu.device, dtype=model.mu.dtype)
            predicted = model.predict(batch) if kind == "kmeans" else model.responsibilities(batch).argmax(dim=1)
            assignments.extend(predicted.cpu().tolist())
    truth = pd.factorize(labels.loc[valid, color_label].map(str))[0]
    return homogeneity_score(truth, assignments)


def summarize_configurations():
    rows = []
    for spec_path, spec in RUN_SPECS:
        dataset, training = spec["dataset"], spec["training"]
        kind = training["model_kind"]
        if kind not in ("kmeans", "hddc"):
            continue
        args = training["arguments"]
        label_field = color_label
        checkpoint = spec_path.parent / ("mfa_model.pt" if kind == "hddc" else "kmeans_model.pt")
        row = {
            "dataset": dataset["id"], "layer": int(dataset["layer"]), "kind": kind,
            "components_total": int(args["K"]), "threshold": args["surgery_threshold"],
            "q_max": args.get("rank") if kind == "hddc" else None, "seed": args["seed"],
            "components_alive": None, "mean_q_k": np.nan, "rank_components": None,
            "checkpoint_status": "missing", "run_id": spec_path.parent.name,
            "checkpoint": str(checkpoint),
            "homogeneity": np.nan, "homogeneity_label": label_field,
        }
        if checkpoint.is_file():
            loader = load_mfa_hddc if kind == "hddc" else load_kmeans
            model = loader(checkpoint, map_location="cpu")
            row.update(component_summary(model, kind))
            row["homogeneity"] = model_homogeneity(model, kind, int(dataset["layer"]))
            print(f"{spec_path.parent.name}: homogeneity={row['homogeneity']:.3f} ({label_field})", flush=True)
            row["checkpoint_status"] = "available"
            del model  # Keep at most one full checkpoint in memory.
        rows.append(row)
    columns = ["dataset", "layer", "kind", "components_total", "threshold", "q_max", "seed",
               "components_alive", "mean_q_k", "rank_components", "homogeneity", "homogeneity_label",
               "checkpoint_status", "run_id", "checkpoint"]
    table = pd.DataFrame(rows, columns=columns)
    for column in ("components_alive", "components_total", "rank_components", "q_max"):
        table[column] = table[column].astype("Int64")
    return table.sort_values(["dataset", "layer", "kind", "components_total", "threshold", "seed", "run_id"]).reset_index(drop=True)


configuration_summary = summarize_configurations()
configuration_table = configuration_summary.copy()
configuration_table["alive / total"] = [
    f"{int(alive) if pd.notna(alive) else '?'} / {int(total)}"
    for alive, total in zip(configuration_table["components_alive"], configuration_table["components_total"])
]
print(f"{len(configuration_summary)} configurations · "
      f"{configuration_summary['checkpoint_status'].eq('available').sum()} saved checkpoints")
display(configuration_table[[
    "dataset", "layer", "kind", "threshold", "q_max", "seed", "alive / total",
    "mean_q_k", "rank_components", "homogeneity", "homogeneity_label", "checkpoint_status", "run_id",
]].style.format({"mean_q_k": "{:.2f}", "homogeneity": "{:.3f}", "threshold": "{:g}"}, na_rep="—"))


hddc__years_neutral__l04__k50__q16__s42__f7c18d6a: homogeneity=0.554 (year)
hddc__years_neutral__l04__k50__q16__s43__75842954: homogeneity=0.558 (year)
hddc__years_neutral__l04__k50__q16__s44__3563c899: homogeneity=0.557 (year)
hddc__years_neutral__l04__k50__q16__s45__0b12d30a: homogeneity=0.559 (year)
hddc__years_neutral__l04__k50__q16__s46__93442129: homogeneity=0.556 (year)
5 configurations · 5 saved checkpoints


,dataset,layer,kind,threshold,q_max,seed,alive / total,mean_q_k,rank_components,homogeneity,homogeneity_label,checkpoint_status,run_id
0,years_neutral,4,hddc,0.01,16,42,50 / 50,13.88,50,0.554,year,available,hddc__years_neutral__l04__k50__q16__s42__f7c18d6a
1,years_neutral,4,hddc,0.01,16,43,50 / 50,14.42,50,0.558,year,available,hddc__years_neutral__l04__k50__q16__s43__75842954
2,years_neutral,4,hddc,0.01,16,44,50 / 50,14.26,50,0.557,year,available,hddc__years_neutral__l04__k50__q16__s44__3563c899
3,years_neutral,4,hddc,0.01,16,45,50 / 50,14.48,50,0.559,year,available,hddc__years_neutral__l04__k50__q16__s45__0b12d30a
4,years_neutral,4,hddc,0.01,16,46,50 / 50,14.40,50,0.556,year,available,hddc__years_neutral__l04__k50__q16__s46__93442129


## Shared model projection: first three local PCs

Each ellipsoid is defined by its component's **first three local covariance PCs**,
then projected into the selected global PCA coordinates. Local PC1/PC2/PC3 keep
these identities when changing the plotting axes.

For HDDC, obtain the leading covariance PCs from an SVD of its loadings, adding
the isotropic noise variance to their eigenvalues. Noise is retained along these
three PCs only. When leading eigenvalues tie (including noise-only directions),
the associated axes are not uniquely determined; the SVD selects a basis.
For KMeans, use the first three saved rank-selected PCs and their eigenvalues.
If fewer than three were saved, show only the available directions; do not invent
missing PCs. Invalid KMeans PCA components retain centers but no footprint.

Only the three scaled local PCs are projected into the fitted 15-PC space.
Models load sequentially, without constructing ambient covariance matrices.
Rerun this section after changing model settings or refitting global PCA.


In [32]:
# Optional plot filters. None includes every value found under ANALYSIS_PATH.
MODEL_K = None
SURGERY_THRESHOLD = None
MODEL_SEED = None


In [33]:
from scipy.stats import chi2


def discover_model_runs():
    matches = {kind: {layer: [] for layer in LAYERS} for kind in MODEL_KINDS}
    rows = []
    for spec_path, spec in RUN_SPECS:
        kind, args = spec["training"]["model_kind"], spec["training"]["arguments"]
        layer = int(spec["dataset"]["layer"])
        if (layer not in results or (MODEL_K is not None and args["K"] != MODEL_K)
                or (MODEL_SEED is not None and args["seed"] != MODEL_SEED)
                or (SURGERY_THRESHOLD is not None and not np.isclose(
                    args["surgery_threshold"], SURGERY_THRESHOLD, rtol=1e-10, atol=0))):
            continue
        filename = "mfa_model.pt" if kind == "hddc" else "kmeans_model.pt"
        checkpoint = spec_path.parent / filename
        if checkpoint.is_file():
            matches[kind][layer].append((spec_path.parent.name, checkpoint))
        rows.append({"layer": layer, "kind": kind, "K": args["K"],
                     "threshold": args["surgery_threshold"], "seed": args["seed"],
                     "available": checkpoint.is_file(), "run": spec_path.parent.name})
    table = pd.DataFrame(rows, columns=["layer", "kind", "K", "threshold", "seed", "available", "run"])
    return matches, table.sort_values(["layer", "kind", "seed"])


def leading_hddc_factors(loadings, psi):
    """Top three covariance PCs, scaled by sqrt(eigenvalue), for isotropic HDDC."""
    loadings = np.asarray(loadings, dtype=np.float64)
    psi = np.asarray(psi, dtype=np.float64)
    if not np.allclose(psi, psi[:, :1], rtol=1e-6, atol=0):
        raise ValueError("Local-PC visualization requires isotropic HDDC noise")
    if np.any(psi <= 0):
        raise ValueError("HDDC noise variances must be positive")
    count, dimension, _ = loadings.shape
    n_pcs = min(3, dimension)
    factors = np.zeros((count, dimension, 3), dtype=np.float64)
    for k, loading in enumerate(loadings):
        vectors, singular_values, _ = np.linalg.svd(loading, full_matrices=False)
        vectors = vectors[:, :n_pcs]
        # Small-q checkpoints may need noise-only PCs outside the loading span.
        for coordinate in range(dimension):
            if vectors.shape[1] == n_pcs:
                break
            direction = np.zeros(dimension)
            direction[coordinate] = 1
            for _ in range(2):
                direction -= vectors @ (vectors.T @ direction)
            norm = np.linalg.norm(direction)
            if norm > 1e-10:
                vectors = np.column_stack([vectors, direction / norm])
        values = np.full(n_pcs, psi[k, 0])
        n_signal = min(n_pcs, len(singular_values))
        values[:n_signal] += singular_values[:n_signal] ** 2
        factors[k, :, :n_pcs] = vectors * np.sqrt(values)
    return factors


def load_projected_model(checkpoint, kind, pca):
    loader = load_mfa_hddc if kind == "hddc" else load_kmeans
    model = loader(checkpoint, map_location="cpu")
    def array(tensor):
        return tensor.detach().cpu().numpy().astype(np.float64)
    with torch.no_grad():
        means = (array(model.mu) - pca.mean_) @ pca.components_.astype(np.float64).T
        ranks = model.component_ranks.detach().cpu().numpy()
        if kind == "hddc":
            factors = leading_hddc_factors(array(model._W()), array(model._psi()))
            valid = np.ones(model.K, dtype=bool)
            weights = array(model.pi_logits.softmax(dim=0))
            details = [f"mixture weight={weight:.5g}" for weight in weights]
            description = "HDDC · first 3 local covariance PCs"
        else:
            n_pcs = min(3, model.q)
            values = array(model.eigenvalues[:, :n_pcs])
            if np.any(values < 0):
                raise ValueError("Negative saved KMeans eigenvalues")
            factors = np.zeros((model.K, model.D, 3), dtype=np.float64)
            factors[:, :, :n_pcs] = array(model.W[:, :, :n_pcs]) * np.sqrt(values)[:, None, :]
            valid = model.pca_valid.detach().cpu().numpy().copy()
            counts = model.cluster_counts.detach().cpu().numpy()
            details = [f"training cluster count={count:,}" for count in counts]
            description = "KMeans · first 3 available saved local PCs"
    local_ranks = (np.linalg.norm(factors, axis=1) > 0).sum(axis=1)
    projected_factors = np.einsum("pd,kdr->kpr", pca.components_.astype(np.float64), factors, optimize=True)
    covariance = projected_factors @ projected_factors.transpose(0, 2, 1)
    if not (np.isfinite(means).all() and np.isfinite(covariance).all()):
        raise ValueError(f"Nonfinite projected geometry: {checkpoint}")
    colors = k3d.helpers.map_colors(
        np.linspace(0.02, 0.98, model.K), k3d.colormaps.matplotlib_color_maps.Turbo, [0, 1]
    )
    hover = [f"component {k}<br>{kind}<br>saved rank={int(ranks[k])}<br>local PCs used={int(local_ranks[k])}<br>{details[k]}"
             + ("<br>no valid PCA footprint" if not valid[k] else "") for k in range(model.K)]
    return {"model": model, "means": means, "covariances": covariance, "valid": valid,
            "local_pc_factors": projected_factors, "local_ranks": local_ranks,
            "colors": colors, "hover": hover, "description": description,
            "checkpoint": str(checkpoint)}


MODEL_KINDS = ("kmeans", "hddc")
matched_runs, run_table = discover_model_runs()
display(run_table)
cluster_geometry = {kind: {} for kind in MODEL_KINDS}
selected_models = {kind: None for kind in MODEL_KINDS}
# Models and geometry are loaded on demand; keep one selected run per family.
print("Use each plot's Run selector to compare saved configurations.")


,layer,kind,K,threshold,seed,available,run
0,4,hddc,50,0.01,42,True,hddc__years_neutral__l04__k50__q16__s42__f7c18d6a
1,4,hddc,50,0.01,43,True,hddc__years_neutral__l04__k50__q16__s43__75842954
2,4,hddc,50,0.01,44,True,hddc__years_neutral__l04__k50__q16__s44__3563c899
3,4,hddc,50,0.01,45,True,hddc__years_neutral__l04__k50__q16__s45__0b12d30a
4,4,hddc,50,0.01,46,True,hddc__years_neutral__l04__k50__q16__s46__93442129


Use each plot's Run selector to compare saved configurations.


## Shared probability and geometry

`RETAINED_PROBABILITY` sets the initial probability. Each plot has its own slider
that resizes its geometry without loading models or refitting PCA. The radius is
`sqrt(chi2.ppf(p, r))`, where `r` is the number of nonzero **local PCs retained**
(three for HDDC; at most three for saved KMeans geometry). The probability belongs
to this truncated local Gaussian before projection, not to the full ambient
Gaussian or mixture. Projection may collapse directions; it does not change the
chosen local probability radius.

The colored segments are the projected local principal semiaxes:
**PC1 red, PC2 green, PC3 blue**. They retain their local ordering, even when the
global projection changes their apparent lengths or angles. `LOCAL_PC_COLORS`
controls these colors. Flat ellipses, lines, and points remain degenerate; no
jitter is added. Component IDs and surface colors are local to each model and
do not imply matching clusters across the two panels.


In [34]:
RETAINED_PROBABILITY = 0.90
ELLIPSE_ALPHA = 0.12
SHOW_CLUSTER_AXES = True
LOCAL_PC_COLORS = (0xE41A1C, 0x2CA02C, 0x377EB8)  # local PC1 red, PC2 green, PC3 blue


def validate_plot_request(pcs, dimension, probability):
    if (len(pcs) != dimension or len(set(pcs)) != dimension
            or any(not isinstance(pc, (int, np.integer)) or not 1 <= pc <= N_COMPONENTS for pc in pcs)):
        raise ValueError(f"Choose {dimension} distinct PCs from 1 to {N_COMPONENTS}")
    if not np.isfinite(probability) or not 0 < probability < 1:
        raise ValueError("Retained probability must be strictly between 0 and 1")
    if not 0 <= ELLIPSE_ALPHA <= 1:
        raise ValueError("ELLIPSE_ALPHA must be between 0 and 1")


def gaussian_footprint(mean, covariance, probability, probability_rank=None):
    """Project a local probability region using its retained local dimension."""
    if not np.isfinite(probability) or not 0 < probability < 1:
        raise ValueError("Retained probability must be strictly between 0 and 1")
    mean = np.asarray(mean, dtype=np.float64)
    covariance = np.asarray(covariance, dtype=np.float64)
    if not (np.isfinite(mean).all() and np.isfinite(covariance).all()):
        raise ValueError("Nonfinite Gaussian geometry")
    values, vectors = np.linalg.eigh((covariance + covariance.T) * 0.5)
    tolerance = max(float(np.max(np.abs(values))) * 1e-9, np.finfo(float).tiny)
    if values.min() < -tolerance:
        raise ValueError("Projected covariance has materially negative eigenvalues")
    keep = np.flatnonzero(values > tolerance)[::-1]
    rank = len(keep)
    if rank == 0:
        return {"rank": 0, "vertices": mean[None], "faces": None, "axes": []}
    degrees = rank if probability_rank is None else probability_rank
    if degrees < rank:
        raise ValueError("Local probability rank cannot be smaller than projected rank")
    transform = vectors[:, keep] * np.sqrt(values[keep] * chi2.ppf(probability, degrees))
    axes = [np.stack([mean - axis, mean + axis]) for axis in transform.T]
    faces = None
    if rank == 1:
        vertices = axes[0]
    elif rank == 2:
        angles = np.linspace(0, 2 * np.pi, 65)
        circle = np.stack([np.cos(angles), np.sin(angles)])
        vertices = mean + (transform @ circle).T
        if len(mean) == 3:
            # Convex planar polygon, triangulated without introducing thickness.
            faces = np.array([[0, j, j + 1] for j in range(1, len(vertices) - 2)], dtype=np.uint32)
    else:
        # Unique poles avoid zero-area triangles in the sphere mesh.
        n_u, n_v = 24, 12
        u = np.arange(n_u) * (2 * np.pi / n_u)
        v = np.arange(1, n_v) * (np.pi / n_v)
        uu, vv = np.meshgrid(u, v)
        rings = np.stack([np.cos(uu) * np.sin(vv), np.sin(uu) * np.sin(vv), np.cos(vv)], axis=-1).reshape(-1, 3)
        sphere = np.vstack([[0, 0, 1], rings, [0, 0, -1]])
        triangles = []
        for j in range(n_u):
            nxt = (j + 1) % n_u
            triangles.append([0, 1 + j, 1 + nxt])
            for ring in range(n_v - 2):
                a, b = 1 + ring * n_u + j, 1 + ring * n_u + nxt
                triangles.extend([[a, a + n_u, b], [b, a + n_u, b + n_u]])
            last = 1 + (n_v - 2) * n_u
            triangles.append([len(sphere) - 1, last + nxt, last + j])
        vertices = mean + sphere @ transform.T
        faces = np.asarray(triangles, dtype=np.uint32)
    return {"rank": rank, "vertices": vertices, "faces": faces, "axes": axes}


def layer_components(kind, layer, pcs, probability):
    geometry = cluster_geometry[kind].get(layer)
    if geometry is None:
        return []
    indices = np.asarray(pcs) - 1
    components = []
    for k, mean in enumerate(geometry["means"][:, indices]):
        covariance = geometry["covariances"][k][np.ix_(indices, indices)]
        local_rank = int(geometry["local_ranks"][k])
        footprint = gaussian_footprint(mean, covariance, probability, local_rank) if geometry["valid"][k] else None
        if footprint is not None:
            radius = np.sqrt(chi2.ppf(probability, local_rank)) if local_rank else 0.0
            projected_axes = geometry["local_pc_factors"][k, indices, :] * radius
            footprint["axes"] = [np.stack([mean - axis, mean + axis]) for axis in projected_axes.T]
        components.append({"id": k, "mean": mean, "color": geometry["colors"][k],
                           "hover": geometry["hover"][k], "footprint": footprint})
    return components


def projection_title(kind, layer, pcs, probability):
    geometry = cluster_geometry[kind].get(layer)
    note = "no model available" if geometry is None else (
        f"{geometry['description']} · K={len(geometry['means'])} · {Path(geometry['checkpoint']).parent.name}"
        f" · {int((~geometry['valid']).sum())} invalid footprints omitted")
    return f"{selection_name} · layer {layer:02d} · PCs {tuple(pcs)}<br><sup>{note} · retained probability {probability:.1%}</sup>"


## Plot helpers

Each explorer has independent layer, PC, point-size, probability, and visibility
controls and its own camera. Both use the fitted activation PCA and ground-truth
colors. Run either plot cell below independently after preparing the models.

Uncheck **Show points** to see only the cluster geometry. Uncheck **Show clusters**
to inspect points alone. Missing models are reported for the selected layer.

Click an ellipsoid to hide that cluster, including its axes and center. Hidden
clusters stay hidden when changing PCs or probability and are remembered per
layer within each plot. **Restore hidden clusters** restores the current layer.

Hover over a Gaussian surface or a component center to show its identity in the
**Last hovered Gaussian** readout above the plot. Component `k` is the zero-based
row `model.mu[k]` in the selected run's checkpoint, with the saved rank and mixture
weight (HDDC) or training cluster count (KMeans). The readout keeps the last hovered
component until the controls change. For line-shaped footprints, hover over the
center. The notebook kernel must be running for hover callbacks.


In [124]:
import ipywidgets as widgets


def k3d_cluster_objects(kind, layer, pcs, probability, hidden, on_hide, on_hover, component_mask=None):
    components = layer_components(kind, layer, pcs, probability)
    if component_mask is not None and components:
        mask = component_mask.detach().cpu().numpy() if torch.is_tensor(component_mask) else np.asarray(component_mask)
        if mask.ndim != 1:
            raise ValueError("Component mask must be a 1D boolean mask or list of component IDs.")
        if mask.dtype == np.bool_:
            if len(mask) != len(components):
                raise ValueError(f"{kind}: boolean mask must have {len(components)} entries for this run.")
            allowed = set(np.flatnonzero(mask))
        else:
            if mask.size and (not np.issubdtype(mask.dtype, np.integer)
                              or np.any(mask < 0) or np.any(mask >= len(components))):
                raise ValueError(f"{kind}: component IDs must be integers from 0 to {len(components) - 1}.")
            allowed = set(mask.tolist())
        components = [c for c in components if c["id"] in allowed]
    components = [c for c in components if c["id"] not in hidden]
    objects = []
    scores = results[layer]["scores"][:, np.asarray(pcs) - 1]
    span = max(float(np.ptp(scores, axis=0).max()), 1e-6)
    for component in components:
        footprint = component["footprint"]
        if footprint is None:
            continue
        vertices = footprint["vertices"].astype(np.float32)
        color = int(component["color"])
        name = component["hover"].replace("<br>", " · ")
        if footprint["faces"] is not None:
            obj = k3d.mesh(vertices, footprint["faces"], color=color, opacity=ELLIPSE_ALPHA,
                           side="double", name=name)
        elif footprint["rank"] == 0:
            obj = k3d.points(vertices, color=color, point_size=span * 0.005, name=name)
        else:
            obj = k3d.line(vertices, color=color, width=span * 0.001, name=name)
        if footprint["faces"] is not None or footprint["rank"] == 0:
            obj.click_callback = lambda event, component_id=component["id"]: on_hide(layer, component_id)
            obj.hover_callback = lambda event, component=component: on_hover(component)
        objects.append(obj)
    if SHOW_CLUSTER_AXES:
        for pc, color in enumerate(LOCAL_PC_COLORS):
            segments = [c["footprint"]["axes"][pc] for c in components
                        if c["footprint"] is not None
                        and np.linalg.norm(c["footprint"]["axes"][pc][1] - c["footprint"]["axes"][pc][0]) > 0]
            if segments:
                vertices = np.concatenate(segments).astype(np.float32)
                indices = np.arange(len(vertices), dtype=np.uint32).reshape(-1, 2)
                objects.append(k3d.lines(vertices, indices, indices_type="segment", color=color,
                                         width=span * 0.0012, name=f"Local PC{pc + 1}"))
    if components:
        centers = k3d.points(np.stack([c["mean"] for c in components]).astype(np.float32),
                             colors=np.array([int(c["color"]) for c in components], dtype=np.uint32),
                             point_size=span * 0.008, name="Component centers")
        # The picked point index addresses the visible list, not the model's rows.
        centers.hover_callback = lambda event: on_hover(components[int(event["index"])])
        objects.append(centers)
    return objects


def make_cluster_explorer(kind, component_mask=None):
    """Create one plot with its own controls, camera, and overlay objects."""
    validate_plot_request((1, 2, 3), 3, RETAINED_PROBABILITY)
    k3d_colors, color_legend = point_coloring()
    modeled_layers = [layer for layer in LAYERS if matched_runs[kind][layer]]
    initial_layer = (modeled_layers or LAYERS)[0]
    layer_control = widgets.Dropdown(options=LAYERS, value=initial_layer, description="Layer")
    run_control = widgets.Dropdown(options=matched_runs[kind][initial_layer] or [("Points only", None)],
                                   description="Run", layout=widgets.Layout(width="95%"))
    pc_options = list(range(1, results[LAYERS[0]]["scores"].shape[1] + 1))
    pc_controls = [widgets.Dropdown(options=pc_options, value=i + 1, description=axis)
                   for i, axis in enumerate(["X PC", "Y PC", "Z PC"])]
    size_control = widgets.FloatLogSlider(value=0.003, base=10, min=-4, max=-1,
                                         step=0.1, description="Point size", continuous_update=False)
    probability_control = widgets.FloatSlider(value=RETAINED_PROBABILITY, min=min(0.01, RETAINED_PROBABILITY),
                                              max=max(0.999, RETAINED_PROBABILITY), step=0.001,
                                              readout_format=".3f", description="Probability", continuous_update=False)
    overlay_control = widgets.Checkbox(value=True, description="Show clusters")
    points_control = widgets.Checkbox(value=True, description="Show points")
    restore_button = widgets.Button(description="Restore hidden clusters", disabled=True,
                                    layout=widgets.Layout(width="200px"))
    hidden_components = defaultdict(set)
    status = widgets.HTML()
    initial = np.asarray(results[initial_layer]["scores"][:, :3], dtype=np.float32)
    cloud = k3d.points(initial.copy(), colors=k3d_colors,
                       shader="3d", opacity=1.0, name=color_label or "Activations",
                       point_size=max(float(np.ptp(initial, axis=0).max()) * size_control.value, 1e-6))
    plot = k3d.plot(height=720, camera_auto_fit=False, mode="callback")
    plot += cloud
    objects = []
    panel_status = widgets.HTML()
    hover_hint = "Hover over a Gaussian surface or component center to identify it (zero-based model index)."
    hover_status = widgets.HTML(value=hover_hint)

    def show_component(component):
        hover_status.value = (
            f"<b>Last hovered Gaussian · layer {layer_control.value:02d}</b><br>"
            f"{component['hover']}<br>"
            f"Model row: <code>selected_models[&quot;{kind}&quot;].mu[{component['id']}]</code>"
        )


    def hide_component(layer, component_id):
        hidden_components[(layer, run_control.value)].add(component_id)
        update_k3d({"owner": restore_button})

    def restore_clusters(button):
        hidden_components[(layer_control.value, run_control.value)].clear()
        update_k3d({"owner": button})

    def update_k3d(change=None):
        nonlocal plot
        pcs = tuple(control.value for control in pc_controls)
        if len(set(pcs)) != 3:
            status.value = "Choose three distinct PCs; the previous projections are still displayed."
            return
        hover_status.value = hover_hint
        probability = probability_control.value
        validate_plot_request(pcs, 3, probability)
        layer = layer_control.value
        checkpoint = run_control.value
        geometry = cluster_geometry[kind].get(layer)
        if checkpoint is None:
            cluster_geometry[kind].clear()
            selected_models[kind] = None
        elif geometry is None or geometry["checkpoint"] != str(checkpoint):
            cluster_geometry[kind].clear()
            cluster_geometry[kind][layer] = load_projected_model(checkpoint, kind, results[layer]["pca"])
        if checkpoint is not None:
            selected_models[kind] = cluster_geometry[kind][layer]["model"]
        indices = np.array(pcs) - 1
        scores = np.asarray(results[layer]["scores"][:, indices], dtype=np.float32)
        ratios = results[layer]["pca"].explained_variance_ratio_[indices]
        point_size = max(float(np.ptp(scores, axis=0).max()) * size_control.value, 1e-6)
        status.value = f"Layer {layer:02d} · PCs {pcs} · {ratios.sum():.1%} explained · {len(scores):,} activations"
        hidden = hidden_components[(layer, checkpoint)]
        status.value += f" · {len(hidden)} hidden clusters"
        restore_button.disabled = not hidden
        owner = change["owner"] if change is not None else None
        cloud.positions = scores
        cloud.point_size = point_size
        cloud.visible = points_control.value
        plot.axes = [f"PC{pc} ({ratio:.1%})" for pc, ratio in zip(pcs, ratios)]
        panel_status.value = projection_title(kind, layer, pcs, probability).replace("<sup>", "").replace("</sup>", "")
        if owner is overlay_control:
            for obj in objects:
                obj.visible = overlay_control.value
        elif owner is not size_control and owner is not points_control:
            # Replace the old geometry when the projection changes.
            for obj in objects:
                plot -= obj
            objects.clear()
            for obj in k3d_cluster_objects(kind, layer, pcs, probability, hidden, hide_component, show_component, component_mask):
                obj.visible = overlay_control.value
                plot += obj
                objects.append(obj)
        if owner is None or owner is layer_control or any(owner is control for control in pc_controls):
            plot.camera_reset()


    def change_layer(change):
        run_control.unobserve(update_k3d, names="value")
        run_control.options = matched_runs[kind][change["new"]] or [("Points only", None)]
        run_control.value = run_control.options[0][1]
        run_control.observe(update_k3d, names="value")
        update_k3d(change)

    layer_control.observe(change_layer, names="value")
    for control in [run_control, *pc_controls, size_control, probability_control, overlay_control, points_control]:
        control.observe(update_k3d, names="value")
    restore_button.on_click(restore_clusters)
    update_k3d()
    return widgets.VBox([
        run_control,
        widgets.HTML(color_legend),
        widgets.HBox([layer_control, *pc_controls]),
        widgets.HBox([size_control, probability_control]),
        widgets.HBox([points_control, overlay_control, restore_button]),
        widgets.HTML("Local covariance axes: " + " &nbsp; ".join(
            f'<span style="color:#{color:06x}"><b>PC{pc + 1}</b></span>'
            for pc, color in enumerate(LOCAL_PC_COLORS))),
        status, panel_status, hover_status, plot,
    ], layout=widgets.Layout(width="100%"))


## Choose components to plot

Set a selection for each model family: `None` shows all components, `[1, 4, 7]`
shows those zero-based component IDs, and `[]` shows none. You can also supply a
boolean NumPy array or Torch tensor with one entry per component (`True` = show).
The mask applies to Gaussian surfaces, local axes, and centers.

Run this cell and rerun the plot cells after editing the masks. IDs refer to the
checkpoint selected in each plot; use a parameter-based mask only with the run
from which it was computed. Changing runs keeps the same mask, and boolean masks
must match the new run's component count. Restore hidden clusters respects this
mask. Activation points remain controlled by **Show points**.


In [176]:
(components_nll < 0).nonzero()  # Check for negative log-likelihoods in the components.
components_nll[4]

tensor(46224.0234)

In [180]:
components_nll

tensor([ 96864.2734, 149726.4688,  23327.5391,  11940.8867,  46224.0234,
         70289.9297,  88095.4375,  30306.2227,   9412.9492,  23117.6504,
         89450.6250, 108859.6172,   4291.9536, 127998.0781,    922.4243,
         25158.1602,  19588.0391,  79407.5625,   9514.8496, -10319.0605,
         97839.8359,  18412.8691,   8881.3311,  81225.4531,   1430.6191,
          5411.2515, 114062.3594,   7300.3726, 137689.7188,  45993.1445,
         23262.7637,  25383.0996,  69516.3281,  26937.4590,  95451.7188,
         77183.5781, 159658.8594,   8013.5327,    390.5137,  33233.5703,
          2243.7417,   2683.9775,  49478.1172,  23786.9062,  84238.0000,
          8775.2676,  99321.4453,  22292.4668, 125592.1250, 104754.1719])

In [189]:
components_nll = torch.tensor(components_nll)
components_mask = ~(components_nll > 5e4)
#components_mask = components_nll < 1e3
components_mask = components_mask.nonzero().reshape(-1).tolist()

/tmp/ipykernel_3977796/2659227380.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  components_nll = torch.tensor(components_nll)


In [ ]:
flattness_mask = flattness_values < 1e-3
flattness_mask = flattness_mask.nonzero().reshape(-1).tolist()



In [192]:
COMPONENT_MASKS = {
    "kmeans": None,  # e.g. [1, 4, 7]
    "hddc": None,    # e.g. [1] to show only Gaussian 1
}

# Example using the independently loaded model (select the same run in the plot):
# COMPONENT_MASKS[KIND] = model.component_ranks <= 5


## KMeans plot


In [155]:
kmeans_explorer = make_cluster_explorer("kmeans", component_mask=COMPONENT_MASKS["kmeans"])
display(kmeans_explorer)


## HDDC plot


In [205]:
hddc_explorer = make_cluster_explorer("hddc", component_mask=COMPONENT_MASKS["hddc"])
display(hddc_explorer)
